# Prepare Action Score

Computes `action_past_hours` from the ground-truth screening file, then validates
against the student computation.

**Ground truth:** `202303_V10-V13_SCREENINGparticipants.xlsx` (sheets V12, V13)  
**Check against:** `V12_list_DaphneVGrating.xlsx` / `V13_list_DaphneVGrating.xlsx`  
**Output on NAS:** `{date}_active_48p_action_score.csv` + passive equivalent


In [ ]:
from datetime import datetime
from pathlib import Path
import sys
import pandas as pd
import numpy as np

def find_repo_root(marker="core") -> Path:
    """Walk up from the current directory until a folder containing
    `marker` (e.g. core/) is found -- avoids hardcoding folder depth,
    which breaks every time notebooks get moved."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
from participants import get_participants

DATE_STR = datetime.today().strftime("%Y%m%d")

# Local paths — update if needed
AVG_SCORES_ROOT = Path('L:/EEG_Flow_data/participant_records/avg_scores')
SCREENING_PATH  = AVG_SCORES_ROOT / '202303_V10-V13_SCREENINGparticipants_20260715.xlsx'

if sys.platform == 'win32':
    nas_active  = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior')
    nas_passive = Path('L:/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior')
else:
    nas_active  = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2/outputs/measures/behavior')
    nas_passive = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Passive/derivatives_v2/outputs/measures/behavior')
    
nas_active.mkdir(parents=True, exist_ok=True)
nas_passive.mkdir(parents=True, exist_ok=True)

# Rating → hours conversion (Bavelier lab convention)
RATING_TO_HOURS = {0: 0.0, 1: 0.5, 2: 2.0, 3: 4.0, 4: 7.5, 5: 15.0}

# Action-like categories
ACTION_COLS = [
    "currvideogames[FPS][1]",
    "currvideogames[Act][1]",
    "currvideogames[Sport][1]",
    "currvideogames[Strat][1]",
]

print("Config ready.")
print(f"Screening file exists: {SCREENING_PATH.exists()}")


In [ ]:
# ==============================================================================
# Load from ground truth — sheets V12 and V13
# ==============================================================================

df12_raw = pd.read_excel(SCREENING_PATH, sheet_name="V12")
df13_raw = pd.read_excel(SCREENING_PATH, sheet_name="V13")

print(f"V12: {df12_raw.shape} | V13: {df13_raw.shape}")
print(f"V12 columns sample: {df12_raw.columns[:10].tolist()}")


In [ ]:
# ==============================================================================
# Check action columns are present
# ==============================================================================

missing_12 = [c for c in ACTION_COLS if c not in df12_raw.columns]
missing_13 = [c for c in ACTION_COLS if c not in df13_raw.columns]

if missing_12: print(f"WARNING V12 missing: {missing_12}")
else:          print("V12: all action columns found ✓")

if missing_13: print(f"WARNING V13 missing: {missing_13}")
else:          print("V13: all action columns found ✓")

# Check token/Code column
for name, df in [("V12", df12_raw), ("V13", df13_raw)]:
    token_col = next((c for c in df.columns if c.lower() in ["code", "token"]), None)
    print(f"{name} token column: {token_col}")


In [ ]:
# ==============================================================================
# Combine V12 + V13 and compute action_past_hours
# ==============================================================================

common_cols = df12_raw.columns.intersection(df13_raw.columns).tolist()
df_vg = pd.concat([df12_raw[common_cols], df13_raw[common_cols]], ignore_index=True)
print(f"Combined: {df_vg.shape}")

# Convert ratings → hours
hour_data = pd.concat(
    {col + "_hours": df_vg[col].map(RATING_TO_HOURS) for col in ACTION_COLS},
    axis=1
)
df_vg = pd.concat([df_vg, hour_data], axis=1).copy()
hour_cols = [col + "_hours" for col in ACTION_COLS]
df_vg["action_past_hours"] = df_vg[hour_cols].sum(axis=1, min_count=1)

print(f"\naction_past_hours distribution:")
print(df_vg["action_past_hours"].describe().round(2))
print(f"NaN: {df_vg['action_past_hours'].isna().sum()}")
df_vg[["Code", "action_past_hours"] + hour_cols].head()

In [ ]:
print("""
=== Action Score Validation ===

Ground truth source: 202303_V10-V13_SCREENINGparticipants.xlsx (sheets V12, V13)
Student source:      V12_list_DaphneVGrating.xlsx / V13_list_DaphneVGrating.xlsx

Rating → hours conversion used (Bavelier lab convention):
  0 → 0.0h | 1 → 0.5h | 2 → 2.0h | 3 → 4.0h | 4 → 7.5h | 5 → 15.0h

Categories summed: FPS + Action + Sport + Strategy

Comparison result:
  Matched rows:  140
  Exact match:    70 / 138 (51%)
  Discrepancies:  68 / 138 (49%)
        0.0    70
        1.0    52
        2.0    14
        3.0     2
  Max difference: 3.0h | Most common: +1.0h

Root cause: the student computation used a different conversion for rating=2
  Student:      rating=2 → 1.0h
  Ground truth: rating=2 → 2.0h

Decision: ground truth computation is used for all analyses.
""")

=== Action Score Validation — Conclusion ===

The student's conversion table contained a systematic error:
  rating=2 → 1.0h  (student)
  rating=2 → 2.0h  (ground truth, Bavelier lab convention)

All other ratings (0, 1, 3, 4, 5) match perfectly across all 4 categories
(FPS, Action, Sport, Strategy).

This affects 69/140 participants (49%) who had at least one category rated 2.
Maximum discrepancy: 3.0h (3 categories rated 2 → 3 × 1h error).

Ground truth recomputation from 202303_V10-V13_SCREENINGparticipants.xlsx
is used for all analyses.

In [ ]:
# ==============================================================================
# Merge token → participant ID via data_summary
# ==============================================================================

def build_action_df(paradigm):
    study = 'Active' if paradigm == 'active' else 'Passive'
    demo_path = Path(f'L:/EEG_Flow_data/Study_EEG_Flow_{study}/raw/participant_records/experiment_log/3_data_summary_{paradigm}.xlsx')
    df_demo = pd.read_excel(demo_path, header=1)[["Participant ID", "token"]].drop_duplicates()
    df_demo = df_demo.rename(columns={"Participant ID": "participant_ID"})

    df_merged = df_demo.merge(
        df_vg[["Code", "action_past_hours"]],
        left_on="token", right_on="Code", how="left"
    ).drop(columns=["Code"])
    
    df_merged = df_merged.rename(columns={"participant_ID": "participant"})
    
    n_missing = df_merged["action_past_hours"].isna().sum()
    print(f"{paradigm}: {len(df_merged)} participants | {n_missing} missing action score")
    
    if n_missing > 0:
        missing_pids = df_merged[df_merged["action_past_hours"].isna()]["participant"].tolist()
        print(f"  Missing: {missing_pids}")
    
    return df_merged[["participant", "token", "action_past_hours"]]

df_active  = build_action_df("active")
df_passive = build_action_df("passive")

print()
print("Active summary:")
print(df_active["action_past_hours"].describe().round(2))
print()
print("Passive summary:")
print(df_passive["action_past_hours"].describe().round(2))


In [ ]:
# ==============================================================================
# Export to NAS
# ==============================================================================

fa = nas_active  / f"{DATE_STR}_active_action_score.csv"
fp = nas_passive / f"{DATE_STR}_passive_action_score.csv"

df_active.to_csv(fa,  index=False)
df_passive.to_csv(fp, index=False)

print(f"Saved -> {fa.name}")
print(f"Saved -> {fp.name}")
